Linda Rynerson
1. Setup a directory for all of our images to go. 

In [81]:
import requests
from pathlib import Path
from torchvision import datasets, transforms
import torch.nn as nn
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

#Choose up to how many photographs we will download of each plant
I_NUM = 500
#dictionary for holding the plant species
plant_species = {
    "jewelweed":"Impatiens capensis",
    "stinging_nettle":"Urtica dioica"
}
base_dir = Path("project_images")
base_dir.mkdir(exist_ok=True)

In [82]:
def download_images(folder_name, scientific_name):
    #create a sub folder for each plant.
    folder = base_dir / folder_name
    folder.mkdir(exist_ok = True)
    #Delete any old files
    for file in folder.glob("*.jpg"):
        file.unlink()
    #Send a request to inaturalist's API for photos with the taxon_name matching the scientific name.
    #Choose research quality grade for 
    response = requests.get("https://api.inaturalist.org/v1/observations",
                               params={
                                   "taxon_name" : scientific_name,
                                   "has[]":"photos",
                                   "quality_grade":"research",
                                   "per_page":I_NUM
                                   
                            }
                            )
    #Check if our request succeeded. 200 means it was successful.
    response.raise_for_status()
    #Convert the json response into a python dictionary. Then get the value stored in the results key and store the list in observations.
    observations = response.json()["results"]
    #Write each of the photos to the folder.
    for i, observation in enumerate(observations[:I_NUM]):
        photo = observation["photos"][0]
        url = photo["url"]
        image = requests.get(url)
        with open(folder /f"{i}.jpg", "wb") as file:
            file.write(image.content)
    
        
        

In [83]:
for folder_name, scientific_name in plant_species.items():
    download_images(folder_name, scientific_name)

2. Build the CNN

In [84]:
#Make all images the same size.
#Use transformations so that the model does not memorize lighting or position.
transform_image = transforms.Compose([transforms.Resize((224, 224)),
                                    transforms.RandomHorizontalFlip(),
                                    transforms.RandomRotation(10),
                                    transforms.ColorJitter(
                                        brightness = 0.1,
                                        contrast = 0.1,
                                        saturation = 0.1
                                    ),
                                    transforms.ToTensor()
                                    ]
                                    )
dataset = datasets.ImageFolder("project_images", transform = transform_image)
print(dataset.class_to_idx)
#80-20 test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator = generator
)
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True
)
test_loader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False
)



{'jewelweed': 0, 'stinging_nettle': 1}


In [85]:
#Build a three layer CNN. Implement max pooling between layers.
class myPlantCNN(nn.Module):
    def __init__(self):
        super(myPlantCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.conv3 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64 * 26 * 26, 120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,2)
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        
        x = self.pool(F.relu(self.conv2(x)))
        
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(-1, 64*26*26)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = myPlantCNN()
print(model)


myPlantCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=43264, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=2, bias=True)
)


3. Define the loss function

In [86]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Train the model

In [87]:
num_epochs = 20
for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        #Clear old gradients
        optimizer.zero_grad()
        #Forward pass
        outputs = model(images)
        #Calculate Loss
        loss = criterion(outputs, labels)
        #Calculate Gradients
        loss.backward()
        #Update model weights
        optimizer.step()
        #Keep track of loss
        running_loss +=loss.item()
    average_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs},"
    f"Loss:{average_loss:.4f}"
    )

Epoch 1/20,Loss:0.7102
Epoch 2/20,Loss:0.6933
Epoch 3/20,Loss:0.6930
Epoch 4/20,Loss:0.6922
Epoch 5/20,Loss:0.6917
Epoch 6/20,Loss:0.6884
Epoch 7/20,Loss:0.6820
Epoch 8/20,Loss:0.6744
Epoch 9/20,Loss:0.6518
Epoch 10/20,Loss:0.6289
Epoch 11/20,Loss:0.5355
Epoch 12/20,Loss:0.4879
Epoch 13/20,Loss:0.5035
Epoch 14/20,Loss:0.4445
Epoch 15/20,Loss:0.4334
Epoch 16/20,Loss:0.3842
Epoch 17/20,Loss:0.3558
Epoch 18/20,Loss:0.3261
Epoch 19/20,Loss:0.3280
Epoch 20/20,Loss:0.3454


Test the model.

In [88]:
correct_test = 0
total_test = 0
actual_labels = []
predicted_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total_test += labels.size(0)
        correct_test +=(predicted == labels).sum().item()
        actual_labels.extend(labels.numpy());
        predicted_labels.extend(predicted.numpy());
        
accuracy = 100 * correct_test/total_test
print(f"Test Accuracy: {accuracy:.2f}%")
correct_train = 0
total_train = 0
model.eval()
with torch.no_grad():
    for images, labels in train_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train +=(predicted == labels).sum().item()
training_accuracy = 100 * correct_train/total_train
print(f"Training Accuracy: {training_accuracy:.2f}%")
# y_true and y_pred are collected numpy arrays or lists
print(classification_report(actual_labels, predicted_labels))

Test Accuracy: 88.89%
Training Accuracy: 89.06%
              precision    recall  f1-score   support

           0       0.93      0.87      0.90        45
           1       0.85      0.92      0.88        36

    accuracy                           0.89        81
   macro avg       0.89      0.89      0.89        81
weighted avg       0.89      0.89      0.89        81



In [89]:
#Print confusion matrix
cm = confusion_matrix(actual_labels, predicted_labels);
print(cm);

[[39  6]
 [ 3 33]]


In [91]:
print(len(train_dataset))
print(len(test_dataset))

320
81
